# 05 Model SARIMAX - Rolling Backtest 24h

## Analysis of `sarimax_2019-2020.ipynb`
1. Uses a short 2019-2020 slice and single forecast window, not full 2025 rolling evaluation.
2. One-shot evaluation for one day is useful for demo, but not for robust operational performance.
3. Feature scope and split strategy differ from your MLForecast/XGBoost benchmark setup.

## Changes in This Notebook
1. Uses the full project range with train `< 2025-01-01` and rolling daily test in 2025.
2. Applies day-ahead `h=24` rolling backtest.
3. Uses aligned exogenous features (including `price_gas`).
4. Exports standardized metrics/predictions for side-by-side comparison.

In [ ]:
# If needed:
# %pip install statsmodels scikit-learn

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [ ]:
# Config
DATA_PATH = '../../data_cleaned/merged/02_4_clean_data.csv'
H = 24
MAX_CYCLES = None  # e.g. 30/60 for debug
ORDER = (1, 0, 1)
SEASONAL_ORDER = (1, 0, 1, 24)

start_date = pd.Timestamp('2019-01-01 00:00:00')
split_date = pd.Timestamp('2025-01-01 00:00:00')
end_date = pd.Timestamp('2026-01-01 00:00:00')


In [ ]:
# Load data
df = pd.read_csv(DATA_PATH)
df['period_start_utc'] = pd.to_datetime(df['period_start_utc'], errors='coerce', utc=True).dt.tz_localize(None)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df[(df['date'] >= start_date) & (df['date'] < end_date)].copy()
df = pd.get_dummies(df, columns=['year'], drop_first=True)
df = df.rename(columns={'period_start_utc': 'ds', 'price': 'y'}).sort_values('ds').reset_index(drop=True)
print(f'Rows: {len(df):,} | Range: {df.ds.min()} -> {df.ds.max()}')


In [ ]:
# Exogenous features aligned with other notebooks
base_exog = [
    'load_forecast_da', 'res_sum_da', 'gen_forecast_da',
    'dayofyear_sin1', 'dayofyear_cos1',
    'hour_sin1', 'hour_cos1',
    'dayofweek_sin1', 'dayofweek_cos1',
    'is_holiday', 'price_gas'
]
year_dummies = [c for c in df.columns if c.startswith('year_')]
exog_cols = base_exog + year_dummies
model_df = df[['ds', 'y'] + exog_cols].copy()
cutoffs = pd.date_range(start=split_date, end=end_date - pd.Timedelta(days=1), freq='D')
if MAX_CYCLES is not None:
    cutoffs = cutoffs[:MAX_CYCLES]
print(f'Cycles: {len(cutoffs)}')


In [ ]:
# Rolling daily backtest
rows = []
failed = 0

for i, cutoff in enumerate(cutoffs, 1):
    train = model_df[model_df['ds'] < cutoff]
    test_h = model_df[(model_df['ds'] >= cutoff) & (model_df['ds'] < cutoff + pd.Timedelta(hours=H))]
    if len(test_h) != H:
        continue
    try:
        m = SARIMAX(
            endog=train['y'],
            exog=train[exog_cols].astype(float),
            order=ORDER,
            seasonal_order=SEASONAL_ORDER,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        r = m.fit(disp=False)
        fc = r.get_forecast(steps=H, exog=test_h[exog_cols].astype(float))
        pred = fc.predicted_mean
        out = test_h[['ds', 'y']].copy()
        out['prediction'] = np.asarray(pred)
        out['cutoff'] = cutoff
        rows.append(out)
    except Exception:
        failed += 1
    if i % 15 == 0:
        print(f'Processed {i}/{len(cutoffs)} cycles | failed={failed}')

results = pd.concat(rows, ignore_index=True).rename(columns={'y': 'actual'})
print(f'Completed cycles: {results.cutoff.nunique()} | Failed cycles: {failed}')
results.head()


In [ ]:
# Metrics
rmse = mean_squared_error(results['actual'], results['prediction'], squared=False)
mae = mean_absolute_error(results['actual'], results['prediction'])
eps = 1e-8
mape = np.mean(np.abs((results['actual'] - results['prediction']) / np.maximum(np.abs(results['actual']), eps))) * 100
print(f'RMSE: {rmse:,.4f}')
print(f'MAE:  {mae:,.4f}')
print(f'MAPE: {mape:,.2f}%')


In [ ]:
# Plot
plot_df = results.set_index('ds').sort_index()
ax = plot_df[['actual']].plot(figsize=(15, 5), title='SARIMAX Rolling Daily Backtest (h=24)')
plot_df['prediction'].plot(ax=ax, alpha=0.85)
ax.legend(['actual', 'prediction'])
plt.show()


In [ ]:
# Unified metrics + export for comparison
from pathlib import Path

# normalize evaluation frame name
if 'results' in locals():
    eval_df = results.copy()
elif 'res' in locals():
    eval_df = res.copy()
else:
    raise ValueError('No results dataframe found (expected `results` or `res`).')

# normalize column names
if 'y' in eval_df.columns and 'actual' not in eval_df.columns:
    eval_df = eval_df.rename(columns={'y': 'actual'})
if 'xgb' in eval_df.columns and 'prediction' not in eval_df.columns:
    eval_df = eval_df.rename(columns={'xgb': 'prediction'})
if 'ds' not in eval_df.columns and eval_df.index.name is not None:
    eval_df = eval_df.reset_index()

required_cols = {'actual', 'prediction'}
missing = required_cols - set(eval_df.columns)
if missing:
    raise ValueError(f'Missing required columns for metrics/export: {missing}')

# robust metrics for power prices (can be near zero/negative)
rmse = mean_squared_error(eval_df['actual'], eval_df['prediction'], squared=False)
mae = mean_absolute_error(eval_df['actual'], eval_df['prediction'])
smape = 100 * np.mean(
    2 * np.abs(eval_df['actual'] - eval_df['prediction']) /
    (np.abs(eval_df['actual']) + np.abs(eval_df['prediction']) + 1e-8)
)

mask = np.abs(eval_df['actual']) >= 10
mape_filtered = (
    np.mean(
        np.abs((eval_df.loc[mask, 'actual'] - eval_df.loc[mask, 'prediction']) /
               np.abs(eval_df.loc[mask, 'actual']))
    ) * 100
    if mask.any() else np.nan
)

print(f'RMSE: {rmse:,.4f}')
print(f'MAE:  {mae:,.4f}')
print(f'sMAPE: {smape:,.2f}%')
print(f'MAPE (|actual|>=10): {mape_filtered:,.2f}%')

out_dir = Path('../../artifacts/model_results')
out_dir.mkdir(parents=True, exist_ok=True)

metrics_df = pd.DataFrame([{
    'model': 'SARIMAX',
    'rmse': rmse,
    'mae': mae,
    'smape': smape,
    'mape_filtered_abs_ge_10': mape_filtered,
    'n_predictions': len(eval_df)
}])

metrics_df.to_csv(out_dir / 'sarimax_metrics_rolling_24h.csv', index=False)

pred_cols = [c for c in ['unique_id', 'cutoff', 'ds', 'actual', 'prediction'] if c in eval_df.columns]
eval_df[pred_cols].to_csv(out_dir / 'sarimax_predictions_rolling_24h.csv', index=False)

metrics_df
